# Deriving QA thresholds

The procedure, run top to bottom: measure the [validation visits](validation-visits.md)
with the current code, derive `WARN`/`FAIL` thresholds from the known-good visits, check
them against the known-bad ones, and review them for adoption. Run it whenever the metrics
code, the visit set or the instrument changes. The rules behind it are in
[`qa-principles.md`](qa-principles.md).

Sections 3 and 4 run `pipetask`. Section 4 is the only one that writes to the Butler (one
new collection), and only when `RUN_PIPELINE` is set. Everything else reads.

**Start Jupyter from a shell with the stack set up**, so that `pipetask` and the Butler are
available to the kernel:

```bash
source /path/to/stack/loadLSST.bash
setup -r /path/to/drp_stella
setup -r /path/to/drp_qa
cd /path/to/drp_qa/docs
jupyter lab qa-thresholds.ipynb
```

Set the inputs in section 1, then run all cells. When it has run, commit it with its
outputs (section 9): they are the record of what the
thresholds were derived from.

## 1. Inputs

The only cell to edit. Name the run once and keep the name: the analysis reads that
collection, and the metrics are cached under its name. Use a new name for every run, and
never a collection that mixes code versions, such as `qaActor/reductions`.

In [ ]:
RUN_NAME = None  # e.g. "20261005": names the output collection; new for every run
RUN_PIPELINE = False  # True runs imageQualityQa into the new collection (section 4)
BUTLER = "/work/datastore"
INPUTS = "drpActor/reductions,PFS/defaults"
FILES = "~/.cache/drp_qa"  # pipetask logs and the metrics cache

What was set up, for the record:

In [ ]:
!eups list --setup drp_stella

In [ ]:
!eups list --setup drp_qa

## Set up

In [ ]:
import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pfs.drp.qa.metrics.calibration import DEFAULT_METRICS, METRIC_SPECS, calibrate, labelRows
from pfs.drp.qa.metrics.readers import readMetrics
from pfs.drp.qa.metrics.validationVisits import loadValidationVisits, unmatchedEntries, visitExpression
from pfs.drp.qa.plotting import plotThresholds

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)

assert RUN_NAME, "set RUN_NAME in the Inputs cell"
COLLECTION = f"u/{getpass.getuser()}/qa-validation/{RUN_NAME}"
PIPELINE = "../pipelines/drpQA.yaml#imageQualityQa"
FILES = Path(FILES).expanduser()
FILES.mkdir(parents=True, exist_ok=True)
STEM = COLLECTION.replace("/", "_")
CACHE = FILES / f"iqQaMetrics-{STEM}.parquet"
METRICS = [*DEFAULT_METRICS, "nLines"]
print(f"collection: {COLLECTION}")
print(f"files:      {FILES}")

## 2. The visits

From the YAML, never typed by hand.

In [ ]:
visitSet = loadValidationVisits()
WHERE = f"instrument = 'PFS' AND {visitExpression(visitSet.visits)}"
print(
    f"{len(visitSet.knownGood)} known_good entries ({len(visitSet.goodVisits)} visits), "
    f"{len(visitSet.confirmedBad)} known_bad, {len(visitSet.unconfirmedBad)} unconfirmed; "
    f"{len(visitSet.visits)} visits"
)
print(WHERE)

## 3. Check the inputs exist

`imageQualityQa` reads the `fitDetectorMap` outputs (`lines`, `detectorMap`) of each visit.
Build the graph without running it. It has one quantum per detector and visit read: up to
about 2,000 for the current set. Far fewer means some visits lack `fitDetectorMap` outputs
in `INPUTS`; find them, or reduce them first. Section 5 lists any visit that ends up with no
metrics.

In [ ]:
qgraphLog = FILES / f"qgraph-{STEM}.log"
!pipetask qgraph -b {BUTLER} -p {PIPELINE} -i {INPUTS} -d "{WHERE}" > {qgraphLog} 2>&1
!grep -iE "quanta|error" {qgraphLog} | tail -n 5

## 4. Run `imageQualityQa`

Only with `RUN_PIPELINE = True` (section 1), and never into an existing collection, so that running all
cells again doesn't start a second run. The full log goes to a file; the notebook keeps its
last lines.

In [ ]:
runLog = FILES / f"pipetask-{STEM}.log"
if RUN_PIPELINE:
    existing = !butler query-collections {BUTLER} {COLLECTION}
    if any(COLLECTION in line for line in existing):
        raise RuntimeError(f"{COLLECTION} exists: choose a new RUN_NAME, or set RUN_PIPELINE = False")
    !pipetask --long-log --log-level PFS=INFO run -j 8 -b {BUTLER} -p {PIPELINE} -i {INPUTS} -o {COLLECTION} -d "{WHERE}" > {runLog} 2>&1
    !tail -n 5 {runLog}
else:
    print(f"RUN_PIPELINE is False: using the existing {COLLECTION}")

## 5. Read the metrics

Read-only, once; later runs read the cache. Delete the cache file to re-read.

In [ ]:
if CACHE.exists():
    metrics = pd.read_parquet(CACHE)
else:
    from lsst.daf.butler import Butler

    butler = Butler(BUTLER, collections=[COLLECTION], writeable=False)
    metrics = readMetrics(butler, visitSet.visits, collections=[COLLECTION])
    metrics.to_parquet(CACHE)
print(f"{len(metrics)} rows over {metrics['visit'].nunique()} visits")
missing = sorted(set(visitSet.visits) - set(metrics["visit"]))
print(f"{len(missing)} validation visits have no metrics: {missing}")

# An entry that matches no row is a check that silently does not run: its
# visits were not processed, or a selector (often seqType) is wrong.
for entry in unmatchedEntries(metrics, visitSet):
    print(
        f"matches no rows: visits {entry.visits[0]}-{entry.visits[-1]} "
        f"seqType={entry.seqType!r} expect={entry.expect}"
    )

## 6. Suggested thresholds

One row per metric and population: arm and observation type; flag rates also per lamp;
line counts per lamp.

In [ ]:
table = calibrate(metrics, visitSet, METRICS)
columns = [
    "metric",
    "group",
    "nGood",
    "nGoodVisits",
    "median",
    "robustRms",
    "warn",
    "fail",
    "failLow",
    "failHigh",
    "goodFlaggedWarn",
    "goodFlaggedFail",
    "reliable",
    "failBounded",
    "degenerate",
    "nBad",
    "badOk",
    "nUnconfirmed",
    "unconfirmedFlagged",
]
table[columns].round(4)

In [ ]:
for _, row in table[table["nBad"] > 0].iterrows():
    print(f"{row['metric']}[{row['group']}]: {row['badSummary']}")

Per population: the CDF of the known-good values, `WARN` (amber) and `FAIL` (red) with
the 95 % interval on the FAIL percentile shaded, and below it the known-bad values (crosses
expect FAIL, triangles WARN) and the unconfirmed ones (open circles). A known-bad marker left
of FAIL (right of it, for `nLines`) is a fault the threshold misses.

In [ ]:
for metric in METRICS:
    labelled = labelRows(metrics, visitSet, metric)
    plotThresholds(labelled, table, metric, absolute=METRIC_SPECS[metric].absolute)
    plt.show()

The provenance sentences, for the `doc` of each threshold adopted:

In [ ]:
for _, row in table[table["nGood"] > 0].iterrows():
    print(f"{row['metric']}[{row['group']}]: warn={row['warn']:g} fail={row['fail']:g}")
    print(f"    {row['provenance']}")

## 7. Review

Go through these in order:

1. **Missing visits and unmatched entries** (section 5). Either is a check that didn't run.
   An entry whose visits are present but which matches nothing has a wrong selector:
   compare its `seqType` with the `seqName` column, character for character.
2. **Known-bad checks.** In each population with known-bad rows, `badOk` must be true. If
   not, look at the panel: a marker on the good side of `FAIL` is a missed fault. Either the
   metric doesn't see this fault, or the visit isn't bad for the reason recorded. Fix
   whichever is wrong, not the threshold.
3. **Sample size.** `reliable` is false below 20 values. `nGoodVisits` is the more honest
   count: the detectors of one visit, and back-to-back exposures, are correlated.
4. **`failBounded`.** When false, the sample is too small to bound the FAIL percentile
   (about 370 values are needed for p99), so `FAIL` sits at the sample's extreme. Usable,
   but expect it to move as data accumulate; the provenance sentence says so.
5. **`degenerate`.** `WARN` is not below `FAIL`, so `WARN` can never fire. It comes from
   ties in the tail; set the pair by hand from the plot.
6. **In-sample flag rates.** `goodFlaggedWarn` and `goodFlaggedFail` should be near 5 % and
   1 %. Much more means ties at the threshold.
7. **Unconfirmed visits.** `unconfirmedFlagged` counts the suspected faults the suggestion
   would flag. Use it to settle them in the YAML ([`validation-visits.md`](validation-visits.md)),
   not to move a threshold.
8. **Populations without good data.** Rows with `nGood` 0 have known-bad visits and nothing
   to compare them with. Add known-good visits for that population, or accept that it goes
   unchecked.

## 8. Adopt

The thresholds are config fields of `ImageQualityQaConfig` in
`python/pfs/drp/qa/imageQualityQa.py`. Changing them changes verdicts, so it is done on a
ticket of its own (for the first derivation, PIPE2D-1917).

| Metric | Fields | Granularity | Populations it gates |
|---|---|---|---|
| `medFwhm` | `fwhmWarnThreshold`, `fwhmFailThreshold` | one value | every arm, except traces (not gated until PIPE2D-1917) |
| `medDxCenter` (absolute) | `dxCenterWarnThreshold`, `dxCenterFailThreshold` | one value | every arm and type |
| `pctFlagged` | `flagRateWarnThreshold`, `flagRateFailThreshold` | per `arm` or `arm:species` | as keyed |

1. Where a field holds one value but this notebook gives one per population, use the most
   lenient suggestion among the populations that field gates. Anything stricter fails good
   detectors in the widest population. Note the populations in the `doc`. The spread
   between populations is what a single value costs: a fault that reaches only the lenient
   population's threshold goes unflagged in the others.
2. Paste the provenance sentence (section 6) into the field's `doc`.
3. Update the threshold tables in the README (`imageQualityQa` → *Pass/Warn/Fail
   Thresholds*) and add a line under `## [Unreleased]` in `CHANGELOG.md`.
4. Run this notebook again with a new `RUN_NAME` and the new defaults, and check that every
   known-good visit passes and every known-bad one gets its expected verdict.

## 9. Commit

Commit this notebook with its outputs on the ticket branch. It shows only validation visits,
which are engineering and calibration visits, so its outputs are fine to publish.

```bash
git add docs/qa-thresholds.ipynb
git commit
```